In [1]:
# add .. path 
import os
import sys
sys.path.append('..')
import utils.llm_training as llm_training
import utils.llm_configs as llm_configs

import logging
import re
from tqdm import tqdm
import numpy as np
from datasets import Dataset
import pandas as pd
import argparse
from sklearn.metrics import roc_auc_score

# --- Basic Configuration ---
dataset ="AMES"
metric="auroc"
model_name="Qwen/Qwen2.5-0.5B"
# Model names: jiosephlee/therapeutic_fine_tuning_1M_v2, jiosephlee/therapeutic_fine_tuning_10M, jiosephlee/therapeutic_fine_tuning_36M

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

os.environ["WANDB_PROJECT"]="medex_fine_tuning"

# --- Load Data and Preprocess---
train_df = pd.read_csv(f'./../data/TDC/{dataset}/train_df.csv')
val_df = pd.read_csv(f'./../data/TDC/{dataset}/val_df.csv')
test_df = pd.read_csv(f'./../data/TDC/{dataset}/test_df.csv')

training_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

log.info(f"Training dataset example: {training_ds[0]}")
log.info(f"Validation dataset example: {val_ds[0]}")
log.info(f"Test dataset example: {test_ds[0]}")

# --- Load Model ---
model_config = llm_configs.ModelConfig(
    id=model_name,
    peft=llm_configs.PeftConfig(
        enabled=False,
        add_eot_token=False,  # No longer doing EOT token for LIMA
    ),
    quantization=llm_configs.QuantizationConfig(mode=None), # Use QLoRA
)

log.info("--- Model Configuration ---")
log.info(model_config.model_dump_json(indent=2))

log.info("\n--- Loading Model for Training ---\n")
model, tokenizer = llm_training.load_model_for_training(model_config, log)

lima_training_config = llm_configs.TrainingConfig(
    run_name = f"{dataset} fine-tuning with {model_name}",
    num_train_epochs = 10,
    learning_rate  = 8e-5,
    logging_strategy = "steps", 
    logging_steps = 1,
    gradient_checkpointing=False,
    context_length = 512,
    use_liger_kernel=True,
    per_device_train_batch_size = 8,
    gradient_accumulation_steps=32,
    warmup_steps  = 0, # If 0, it does not override warmup ratio
    warmup_ratio = 0.1, # Use our default warmup ratio instead
    packing=True,
    padding_free = True,
    sequential_sampling = False,
    reverse_ffd_packing= False,
    remove_unused_columns=False,
)

log.info(f"\n--- Starting {dataset} Fine-Tuning ---")
trainer = llm_training.sft_train_on_dataset(
    model=model,
    tokenizer=tokenizer,
    log=log,
    train_dataset=training_ds,
    train_cfg=lima_training_config,
    train=True,
    use_liger_loss = True
)

log.info("\n\n--- Fine-Tuning Complete ---\n\n")
log.info(f"Training arguments: {trainer.args}")


2025-07-12 17:29:27 - INFO - [__main__] - Training dataset example: {'text': 'Q: This is the SMILES string of the drug: Nc1cccc([N+](=O)[O-])c1CO. Is this drug mutagenic?\nA: No'}
2025-07-12 17:29:27 - INFO - [__main__] - Validation dataset example: {'text': 'Q: This is the SMILES string of the drug: O=[N+]([O-])c1ccc(-c2nc3n(c2[N+](=O)[O-])CCS3)cc1. Is this drug mutagenic?\nA: ', 'Y': 1}
2025-07-12 17:29:27 - INFO - [__main__] - Test dataset example: {'text': 'Q: This is the SMILES string of the drug: CC(=O)Nc1ccc2c(=O)c(=O)c3cccc4ccc1c2c43. Is this drug mutagenic?\nA: ', 'Y': 1}
2025-07-12 17:29:27 - INFO - [__main__] - --- Model Configuration ---
2025-07-12 17:29:27 - INFO - [__main__] - {
  "id": "Qwen/Qwen2.5-0.5B",
  "torch_dtype": "auto",
  "attn_implementation": "flash_attention_2",
  "peft": {
    "enabled": false,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gat

Adding EOS to train dataset:   0%|          | 0/5094 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5094 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/5094 [00:00<?, ? examples/s]

2025-07-12 17:29:29 - INFO - [liger_kernel.transformers.monkey_patch] - Applying Liger kernels to model instance with model type: qwen2 with kwargs: {}


...

wandb: Currently logged in as: jiosephlee (upenn-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
1,2.996700
2,2.999200
3,2.663400
4,2.876700
5,1.460800
6,1.193200
7,1.023900
8,0.968100
9,0.915400
10,0.862700


2025-07-12 17:37:12 - INFO - [__main__] - SFT training complete.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
train/global_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
train/grad_norm,██▆▇▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▁▅████▇▇▆▆▅▅▄▃▃▂▂▁▁▁
train/loss,██▇█▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
train/num_tokens,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
total_flos,4721399051466240.0
train/epoch,10
train/global_step,20
train/grad_norm,1.52344
train/learning_rate,0.0


2025-07-12 17:37:13 - INFO - [__main__] - 

--- Fine-Tuning Complete ---


2025-07-12 17:37:13 - INFO - [__main__] - Training arguments: SFTConfig(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_p

In [ ]:
for i, batch in enumerate(trainer.get_train_dataloader()):
    print(batch)

{'input_ids': tensor([[    48,     25,   1096,  ...,     25,   7414, 151643]],
       device='cuda:0'), 'position_ids': tensor([[ 0,  1,  2,  ..., 38, 39, 40]], device='cuda:0'), 'labels': tensor([[    48,     25,   1096,  ...,     25,   7414, 151643]],
       device='cuda:0')}
{'input_ids': tensor([[    48,     25,   1096,  ...,     25,   2308, 151643]],
       device='cuda:0'), 'position_ids': tensor([[ 0,  1,  2,  ..., 76, 77, 78]], device='cuda:0'), 'labels': tensor([[    48,     25,   1096,  ...,     25,   2308, 151643]],
       device='cuda:0')}
{'input_ids': tensor([[    48,     25,   1096,  ...,     25,   2308, 151643]],
       device='cuda:0'), 'position_ids': tensor([[ 0,  1,  2,  ..., 47, 48, 49]], device='cuda:0'), 'labels': tensor([[    48,     25,   1096,  ...,     25,   2308, 151643]],
       device='cuda:0')}
{'input_ids': tensor([[    48,     25,   1096,  ...,     25,   7414, 151643]],
       device='cuda:0'), 'position_ids': tensor([[ 0,  1,  2,  ..., 32, 33, 34]], de

In [3]:
# --- Evaluate ---
log.info("\n\n--- Evaluating ---\n\n")

inference_cfg = llm_configs.InferenceConfig(
    temperature=0,
    do_sample=False,
    repetition_penalty=1.0,
    max_new_tokens=64,
)

targets, preds = [], []

for i in tqdm(range(len(test_ds)), desc="Inference on test set"):
    row = test_ds[i]
    prompt = row["text"]
    gt_answer = "yes" if row["Y"] == 1 else "no"
    
    gen_text = llm_training.generate_text(model, tokenizer, prompt, inference_cfg)
    
    # Extract generated text (remove the prompt part)
    generated_response = gen_text[len(prompt):].strip().lower()

    if i < 10:
        print(f"Prompt: {prompt}")
        print(f"Generated response: {gen_text}")
        print(f"GT answer: {gt_answer}")
        print("-"*100)
    if i == 10:
        print(llm_training.analyze_text_generation(model, tokenizer, prompt, 'cuda', 8))
        break
    # Simple matching - check if "yes" or "no" appears in the response
    if "yes" in generated_response:
        pred_answer = "yes"
    elif "no" in generated_response:
        pred_answer = "no"
    else:
        # If neither yes nor no is found, skip this example
        continue

    
    targets.append(gt_answer)
    preds.append(pred_answer)

# ------------------
# Compute Accuracy
# ------------------
targets = np.array(targets)
preds = np.array(preds)

if metric == "accuracy":
    accuracy = np.mean(targets == preds)
    print(f"\nAccuracy on {len(targets)} examples: {accuracy:.4f}")
elif metric == "auroc":
    auroc = roc_auc_score(targets, preds)
    print(f"\nAUROC on {len(targets)} examples: {auroc:.4f}")

# Save model before we LIMA tune
#model.push_to_hub('jiosephlee/therapeutic_fine_tuning_36M')
#tokenizer.push_to_hub('jiosephlee/therapeutic_fine_tuning_36M')

2025-07-12 17:43:53 - INFO - [__main__] - 

--- Evaluating ---


Inference on test set:   0%|          | 7/1457 [00:00<00:45, 32.16it/s]

Prompt: Q: This is the SMILES string of the drug: CC(=O)Nc1ccc2c(=O)c(=O)c3cccc4ccc1c2c43. Is this drug mutagenic?
A: 
Generated response: Q: This is the SMILES string of the drug: CC(=O)Nc1ccc2c(=O)c(=O)c3cccc4ccc1c2c43. Is this drug mutagenic?
A: 1<|endoftext|>
GT answer: yes
----------------------------------------------------------------------------------------------------
Prompt: Q: This is the SMILES string of the drug: O=c1c(=O)c2c([N+](=O)[O-])ccc3ccc4cccc1c4c32. Is this drug mutagenic?
A: 
Generated response: Q: This is the SMILES string of the drug: O=c1c(=O)c2c([N+](=O)[O-])ccc3ccc4cccc1c4c32. Is this drug mutagenic?
A: 1<|endoftext|>
GT answer: yes
----------------------------------------------------------------------------------------------------
Prompt: Q: This is the SMILES string of the drug: O=c1c(=O)c2ccc([N+](=O)[O-])c3ccc4cccc1c4c32. Is this drug mutagenic?
A: 
Generated response: Q: This is the SMILES string of the drug: O=c1c(=O)c2ccc([N+](=O)[O-])c3ccc4cccc1c4c32

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Inference on test set:   1%|          | 10/1457 [00:00<00:53, 26.83it/s]

Prompt: Q: This is the SMILES string of the drug: C1O[C@H]1[C@H]1CO1. Is this drug mutagenic?
A: 
Generated response: Q: This is the SMILES string of the drug: C1O[C@H]1[C@H]1CO1. Is this drug mutagenic?
A: 1<|endoftext|>
GT answer: yes
----------------------------------------------------------------------------------------------------
Prompt: Q: This is the SMILES string of the drug: C1OC1C1CO1. Is this drug mutagenic?
A: 
Generated response: Q: This is the SMILES string of the drug: C1OC1C1CO1. Is this drug mutagenic?
A: 1<|endoftext|>
GT answer: yes
----------------------------------------------------------------------------------------------------
Prompt: Q: This is the SMILES string of the drug: CC1(C2CO2)CO1. Is this drug mutagenic?
A: 
Generated response: Q: This is the SMILES string of the drug: CC1(C2CO2)CO1. Is this drug mutagenic?
A: 1<|endoftext|>
GT answer: yes
----------------------------------------------------------------------------------------------------
Output: Q: T

ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.